# Hate Speech Detection - Training Pipeline

This notebook covers the data preparation, baseline training, and BERT fine-tuning.

In [5]:
import os
import pandas as pd
import sys
sys.path.append('../src')

from ml.preprocessing import download_dataset, prepare_data
from ml.baseline import train_baseline
from ml.bert_train import run_training

print("Environment Ready")

Environment Ready


In [6]:
## 1. Data Preparation
Download and clean the Davidson dataset.

SyntaxError: invalid syntax (3977810033.py, line 2)

In [3]:
DATA_URL = "https://raw.githubusercontent.com/t-davidson/hate-speech-and-offensive-language/master/data/labeled_data.csv"
RAW_DATA_PATH = "../data/labeled_data.csv"

if not os.path.exists('../data'):
    os.makedirs('../data')

download_dataset(DATA_URL, RAW_DATA_PATH)
prepare_data(RAW_DATA_PATH, output_dir='../data')

Dataset already exists.
Cleaning text...
Data split complete: Train(17348), Val(3717), Test(3718)


(                                                    text  label
 14336  rt the afl now has an american draft system be...      2
 14047  rt mom brought me mcdonalds for lunch i had al...      1
 18468  rt imma cool as bitch so if a bitch dont like ...      1
 8442                                  charlie xcx is bae      2
 1290        8220 lmao lexy such a hoe 1285148221 lmfaooo      1
 ...                                                  ...    ...
 13952  put the redskins in your starting lineup inste...      2
 11848     is hillary the 8216dodo bird8217 candidate via      2
 152    can pornhub just get a gaming stream feature s...      1
 3794   mrcoxthe chase me charlie of his daybend over ...      1
 1638   8220 8220 how i lose this sweet8221you aint lo...      1
 
 [17348 rows x 2 columns],
                                                     text  label
 6101   kept the border open to disease from central a...      2
 19961                            rt kick her in the cunt    

## 2. Baseline Model (TF-IDF + Logistic Regression)
Establishing a performance baseline.

In [7]:
# Note: Modify src/ml/baseline.py to accept paths or just run it as is
%run ../src/ml/baseline.py

Error: train.csv not found. Please run preprocessing.py first.


## 3. BERT Fine-tuning
Fine-tuning BERT-base-uncased for 3-class classification.

In [8]:
# This might take time if running on CPU
run_training()

Using device: cpu


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/Users/emirnar/.pyenv/versions/3.11.9/lib/python3.11/site-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 1/3


100%|█████████████████████████████████████████| 125/125 [03:17<00:00,  1.58s/it]


Train loss 0.5209256374835968 accuracy 0.817
Val loss 0.4016531265806407 accuracy 0.862
Epoch 2/3


100%|█████████████████████████████████████████| 125/125 [03:30<00:00,  1.69s/it]


Train loss 0.2970626953989267 accuracy 0.9035
Val loss 0.3561341440072283 accuracy 0.9
Epoch 3/3


100%|█████████████████████████████████████████| 125/125 [03:44<00:00,  1.80s/it]


Train loss 0.23306972352415323 accuracy 0.9255
Val loss 0.3647744944319129 accuracy 0.896
Training complete. Best Val Accuracy: tensor(0.9000, dtype=torch.float64)


## 4. Inference Test
Test the trained model locally.

In [14]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification

model_path = '../models/bert_hate_speech'
if os.path.exists(model_path):
    tokenizer = BertTokenizer.from_pretrained(model_path)
    model = BertForSequenceClassification.from_pretrained(model_path)
    
    text = "That is great!"
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    pred = torch.argmax(probs, dim=1).item()
    
    labels = {0: "Hate Speech", 1: "Offensive Language", 2: "Neutral"}
    print(f"Text: {text}")
    print(f"Prediction: {labels[pred]}")
else:
    print("BERT model not found. Train it first!")

Text: That is great!
Prediction: Neutral
